# 1 - Crear Spark Session

In [28]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailStreamingLambda")
    .config(
        "spark.jars",
        "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/spark-token-provider-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/kafka-clients-3.6.1.jar,/opt/spark/jars/commons-pool2-2.11.1.jar"
    )
    .getOrCreate()
)

spark.version

'3.5.1'

In [29]:
!ls /opt/spark/jars/

commons-pool2-2.11.1.jar     spark-sql-kafka-0-10_2.12-3.5.1.jar
kafka-clients-3.6.1.jar      spark-token-provider-kafka-0-10_2.12-3.5.1.jar
mysql-connector-j-8.4.0.jar


# 2 - Verificar conexión con Kafka

In [30]:
kafka_server = "kafka:9092"

print(kafka_server)

kafka:9092


# 3 - Leer orders desde Kafka

In [31]:
orders_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "orders_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

orders_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



# 4 - Convertir JSON de orders

In [32]:
from pyspark.sql.types import *

orders_schema = StructType([
    StructField(
        "order_id",
        IntegerType()
    ),
    StructField(
        "order_date",
        TimestampType()
    ),
    StructField(
        "order_customer_id",
        IntegerType()
    ),
    StructField(
        "order_status",
        StringType()
    )
])

### Trasformamos

In [33]:
from pyspark.sql.functions import *


orders_stream = (
    orders_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            orders_schema
        ).alias("data")
    )
    .select("data.*")
)


orders_stream.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



# 5 - Mostrar streaming de orders

In [34]:
query_orders = (
    orders_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/12 05:12:11 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b50b643e-f5f0-4df1-9dea-4e7205b84d8d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/12 05:12:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [35]:
query_orders.stop()

# 6 - Leer order_items

In [36]:
items_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "order_items_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

In [37]:
items_schema = StructType([

    StructField(
        "order_item_id",
        IntegerType()
    ),

    StructField(
        "order_item_order_id",
        IntegerType()
    ),

    StructField(
        "order_item_product_id",
        IntegerType()
    ),

    StructField(
        "order_item_quantity",
        IntegerType()
    ),

    StructField(
        "order_item_subtotal",
        DoubleType()
    ),

    StructField(
        "order_item_product_price",
        DoubleType()
    )
])

In [38]:
items_stream = (
    items_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            items_schema
        ).alias("data")
    )
    .select("data.*")
)


items_stream.printSchema()

root
 |-- order_item_id: integer (nullable = true)
 |-- order_item_order_id: integer (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: integer (nullable = true)
 |-- order_item_subtotal: double (nullable = true)
 |-- order_item_product_price: double (nullable = true)



# 7 - Mostrar items

In [39]:
query_items = (
    items_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/12 05:12:11 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-11bfe22a-0dc2-4dd8-87e9-baae19278cdc. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/12 05:12:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [40]:
query_items.stop()

26/08/12 05:12:11 WARN Shell: Interrupted while joining on: Thread[Thread-10876,5,main]
java.lang.InterruptedException
	at java.base/java.lang.Object.wait(Native Method)
	at java.base/java.lang.Thread.join(Thread.java:1313)
	at java.base/java.lang.Thread.join(Thread.java:1381)
	at org.apache.hadoop.util.Shell.joinThread(Shell.java:1042)
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1002)
	at org.apache.hadoop.util.Shell.run(Shell.java:900)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1212)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1306)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1288)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSy

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+



In [41]:
from pyspark.sql.functions import *
# "La columna order_date representa el tiempo del evento. Permite que lleguen datos atrasados hasta 10 minutos."
orders_stream_watermark = (
    orders_stream
    .withWatermark(
        "order_date",
        "2 minutes"
    )
)

In [42]:
#"Para este stream, considera válidos los eventos atrasados hasta 10 minutos usando event_time."
items_stream_watermark = (
    items_stream
    .withColumn(
        "event_time",
        current_timestamp()
    )
    .withWatermark(
        "event_time",
        "2 minutes"
    )
)

# Revisa un procesamiento en medio 

In [43]:
join_condition = (
    orders_stream_watermark.order_id ==
    items_stream_watermark.order_item_order_id
)

In [44]:
sales_stream = (
    orders_stream_watermark
    .join(
        items_stream_watermark,
        join_condition,
        "inner"
    )
)

In [45]:
ventas = (
    sales_stream
    .select(
        orders_stream_watermark.order_id,
        orders_stream_watermark.order_customer_id,
        orders_stream_watermark.order_status,
        items_stream_watermark.order_item_product_id,
        items_stream_watermark.order_item_quantity,
        items_stream_watermark.order_item_subtotal
    )
)

In [46]:
ventas.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: integer (nullable = true)
 |-- order_item_subtotal: double (nullable = true)



In [47]:
query_join = (
    ventas
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/12 05:12:12 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d34fa32e-8a4e-4b21-b601-ee6e7408020a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/12 05:12:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [48]:
query_join.stop()

26/08/12 05:12:12 WARN Shell: Interrupted while joining on: Thread[Thread-10920,5,main]
java.lang.InterruptedException
	at java.base/java.lang.Object.wait(Native Method)
	at java.base/java.lang.Thread.join(Thread.java:1313)
	at java.base/java.lang.Thread.join(Thread.java:1381)
	at org.apache.hadoop.util.Shell.joinThread(Shell.java:1042)
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1002)
	at org.apache.hadoop.util.Shell.run(Shell.java:900)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1212)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1306)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1288)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSy

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|
+--------+-----------------+------------+---------------------+-------------------+-------------------+
+--------+-----------------+------------+---------------------+-------------------+-------------------+



In [49]:
join_query = (
    ventas
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .trigger(
        processingTime="5 seconds"
    )
    .start()
)

26/08/12 05:12:44 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3293a9ce-a82c-4718-a898-46492b2004eb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/12 05:12:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [50]:
ventas_debug_raw = (
    orders_stream_watermark
    .join(
        items_stream_watermark,
        orders_stream_watermark.order_id ==
        items_stream_watermark.order_item_order_id,
        "inner"
    )
)

streaming_query = (
    ventas
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "path",
        "hdfs://namenode:8020/lambda/speed"
    )
    .option(
        "checkpointLocation",
        "hdfs://namenode:8020/lambda/checkpoint_speed"
    )
    .trigger(
        processingTime="2 minutes"
    )
    .start()
)

26/08/12 05:12:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [51]:
for s in spark.streams.active:
    print(s.name, s.id, s.status)

None b25d087c-091c-4bcf-9346-e6843a3b001b {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
None b7f69dab-7acb-4dff-b1d2-4dcc31d6e657 {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
None 0a76c494-398c-40ef-a641-a15ea0a2a91c {'message': 'Getting offsets from KafkaV2[Subscribe[orders_topic]]', 'isDataAvailable': False, 'isTriggerActive': True}


In [52]:
print("streaming_query.id:", streaming_query.id)

for s in spark.streams.active:
    if s.id != streaming_query.id:
        print("Deteniendo stream viejo:", s.id)
        s.stop()

print("--- Streams activos después de limpiar ---")
for s in spark.streams.active:
    print(s.id, s.status)

streaming_query.id: 0a76c494-398c-40ef-a641-a15ea0a2a91c
Deteniendo stream viejo: b25d087c-091c-4bcf-9346-e6843a3b001b
Deteniendo stream viejo: b7f69dab-7acb-4dff-b1d2-4dcc31d6e657
--- Streams activos después de limpiar ---
0a76c494-398c-40ef-a641-a15ea0a2a91c {'message': 'Getting offsets from KafkaV2[Subscribe[orders_topic]]', 'isDataAvailable': False, 'isTriggerActive': True}


26/08/12 05:12:44 ERROR TorrentBroadcast: Store broadcast broadcast_84 fail, remove all pieces of the broadcast
[Stage 13:===========================================>          (160 + 2) / 200]